In [1]:
import cohere
import pandas as pd

In [3]:
ckey = "Y1ynMJK4f01KHehKQh3PLwlsHpZSaRc29XNrYuKW"
co = cohere.Client(ckey)

In [3]:
three_words = pd.DataFrame({'text':
  [
      'joy',
      'happiness',
      'potato'
  ]})

three_words

,text
0,joy
1,happiness
2,potato


In [4]:
three_words_emb = co.embed(texts=list(three_words['text']),
                           model='embed-english-v2.0').embeddings

In [5]:
word_1 = three_words_emb[0]
word_2 = three_words_emb[1]
word_3 = three_words_emb[2]

In [6]:
len(word_1)

4096

In [3]:
sentences = pd.DataFrame({'text':
  [
   'Where is the world cup?',
   'The world cup is in Qatar',
   'What color is the sky?',
   'The sky is blue',
   'Where does the bear live?',
   'The bear lives in the the woods',
   'What is an apple?',
   'An apple is a fruit',
   'What is Appian Fourth quarter cloud subscription revenue?',
   'Appian Fourth quarter cloud subscription revenue increased 26% year-over-year to $83.1 million'
  ]})

sentences

,text
0,Where is the world cup?
1,The world cup is in Qatar
2,What color is the sky?
3,The sky is blue
4,Where does the bear live?
5,The bear lives in the the woods
6,What is an apple?
7,An apple is a fruit
8,What is Appian Fourth quarter cloud subscripti...
9,Appian Fourth quarter cloud subscription reven...


In [4]:
emb = co.embed(texts=list(sentences['text']),
               model='embed-english-v2.0').embeddings

# Explore the 10 first entries of the embeddings of the 3 sentences:
for e in emb:
    print(e[:3])

[0.27148438, -0.3786621, -1.0263672]
[0.4987793, 1.2255859, 0.40698242]
[-0.23486328, -0.9375, 0.9614258]
[0.08538818, -0.31958008, 0.93066406]
[0.49536133, -0.34985352, -1.6171875]
[1.2285156, -1.3798828, -1.8388672]
[0.15454102, -0.921875, 1.5996094]
[1.0751953, -0.71972656, 0.92822266]
[0.013038635, -0.24316406, 2.8925781]
[0.3869629, -0.53759766, 1.0712891]


In [5]:
len(emb[0])

4096

In [6]:
from utils import umap_plot

In [7]:
chart = umap_plot(sentences, emb)

c:\users\ankan\appdata\local\programs\python\python37\lib\site-packages\sklearn\manifold\_spectral_embedding.py:261: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  "Graph is not fully connected, spectral embedding may not work as expected."


In [8]:
chart.interactive()

alt.Chart(...)

In [4]:
from annoy import AnnoyIndex
import numpy as np
import pandas as pd
import re

In [10]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""

In [11]:
# Split into a list of sentences
texts = text.split('.')

# Clean up to remove empty spaces and new lines
texts = np.array([t.strip(' \n') for t in texts])
texts

array(['Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan',
       'It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine',
       'Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind',
       'Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007',
       'Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar',
       'Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm',
       'Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles',

In [12]:
# Split into a list of paragraphs
texts = text.split('\n\n')

# Clean up to remove empty spaces and new lines
texts = np.array([t.strip(' \n') for t in texts])
texts

array(['Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.\nIt stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.\nSet in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.',
       'Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.\nCaltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.\nCinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.\nPrincipal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.\nInterstellar uses extensive practical 

In [13]:
# Split into a list of sentences
texts = text.split('.')

# Clean up to remove empty spaces and new lines
texts = np.array([t.strip(' \n') for t in texts])
texts

array(['Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan',
       'It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine',
       'Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind',
       'Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007',
       'Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar',
       'Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm',
       'Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles',

In [14]:
title = 'Interstellar (film)'

texts = np.array([f"{title} {t}" for t in texts])
texts

array(['Interstellar (film) Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan',
       'Interstellar (film) It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine',
       'Interstellar (film) Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind',
       'Interstellar (film) Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007',
       'Interstellar (film) Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar',
       'Interstellar (film) Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format

In [15]:
response = co.embed(
    texts=texts.tolist()
).embeddings

default model on embed will be deprecated in the future, please specify a model in the request.


In [16]:
embeds = np.array(response)
embeds.shape

(15, 4096)

In [17]:
search_index = AnnoyIndex(embeds.shape[1], 'angular')
# Add all the vectors to the search index
for i in range(len(embeds)):
    search_index.add_item(i, embeds[i])

search_index.build(10) # 10 trees
search_index.save('test.ann')

True

In [18]:
pd.set_option('display.max_colwidth', None)

def search(query):

  # Get the query's embedding
  query_embed = co.embed(texts=[query]).embeddings

  # Retrieve the nearest neighbors
  similar_item_ids = search_index.get_nns_by_vector(query_embed[0],
                                                    3,
                                                  include_distances=True)
  # Format the results
  results = pd.DataFrame(data={'texts': texts[similar_item_ids[0]],
                              'distance': similar_item_ids[1]})

  print(texts[similar_item_ids[0]])
    
  return results

In [19]:
query = "How much did the film make?"
results = search(query)

default model on embed will be deprecated in the future, please specify a model in the request.


['Interstellar (film) The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014'
 'Interstellar (film) Interstellar premiered on October 26, 2014, in Los Angeles'
 'Interstellar (film) In the United States, it was first released on film stock, expanding to venues using digital projectors']


In [5]:
def rerank_responses(query, responses, num_responses=10):
    reranked_responses = co.rerank(
        model = 'rerank-english-v2.0',
        query = query,
        documents = responses,
        top_n = num_responses,
        )
    return reranked_responses

In [21]:
texts = results["texts"]
reranked_text = rerank_responses(query, texts)

In [22]:
print(query)
for i, rerank_result in enumerate(reranked_text):
    print(f"i:{i}")
    print(f"{rerank_result}")
    print()

How much did the film make?
i:0
RerankResult<document['text']: Interstellar (film) The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014, index: 0, relevance_score: 0.5244708>

i:1
RerankResult<document['text']: Interstellar (film) In the United States, it was first released on film stock, expanding to venues using digital projectors, index: 2, relevance_score: 0.0658447>

i:2
RerankResult<document['text']: Interstellar (film) Interstellar premiered on October 26, 2014, in Los Angeles, index: 1, relevance_score: 0.020527788>



In [98]:
question = "What is the Full Year 2024 guidance for Adjusted EBITDA income expected of Appian?"

In [99]:
text = """
PG*** Appian Corporation.

PG*** Appian Corporation. Fourth quarter cloud subscription revenue increased 26% year-over-year to $83.1 million. Full year cloud subscription revenue increased 29% year-over year to $304.5 million.

PG*** MCLEAN, Va., Feb. 15, 2024 (GLOBE NEWSWIRE) -- Appian (Nasdaq: APPN) today announced financial results for the fourth quarter and full year ended December 31, 2023.

CS*** “Appian delivered our plan in 2023 and reached two milestones. Full year revenue exceeded half a billion dollars, and we achieved the highest quarterly gross margin in our public history,” said Matt Calkins, CEO & Founder.


SCHQ*** Fourth Quarter 2023 Financial Highlights:.

PG*** Revenue: Cloud subscription revenue was $83.1 million, up 26% compared to the fourth quarter of 2022. Total subscriptions revenue, which includes sales of our cloud subscriptions, on-premises term license subscriptions, and maintenance and support, increased 24% year-over-year to $115.8 million. Professional services revenue was $29.5 million, a decrease of 9% compared to the fourth quarter of 2022. Total revenue was $145.3 million, up 16% compared to the fourth quarter of 2022. Cloud subscription revenue retention rate was 119% as of December 31, 2023.

PG*** Operating loss and non-GAAP operating loss: GAAP operating loss was $(16.8) million, compared to $(40.6) million for the fourth quarter of 2022. Non-GAAP operating loss was $(1.4) million, compared to $(26.8) million for the fourth quarter of 2022.

PG*** Net loss and non-GAAP net loss: GAAP net loss was $(10.0) million, compared to $(34.4) million for the fourth quarter of 2022. GAAP net loss per share was $(0.14) for the fourth quarter of 2023, compared to $(0.47) for the fourth quarter of 2022. Non-GAAP net income was $4.9 million, compared to non-GAAP net loss of $(20.6) million for the fourth quarter of 2022. Non-GAAP net income per diluted share was $0.06, compared to the $(0.28) net loss per share for the fourth quarter of 2022. GAAP net loss and non-GAAP net income for the fourth quarter of 2023 included $11.1 million, or $0.15 per share, of foreign currency exchange gains. GAAP and non-GAAP net loss for the fourth quarter of 2022 included $8.5 million, or $0.12 per share, of foreign currency exchange gains. We do not forecast foreign exchange rate movements.

PG*** Adjusted EBITDA: Adjusted EBITDA was $1.0 million, compared to adjusted EBITDA loss of $(24.8) million for the fourth quarter of 2022.

PG*** Cash flows: Net cash used in operating activities was $(8.2) million for the three months ended December 31, 2023 compared to $(12.6) million of net cash used in operating activities for the same period in 2022.

PG*** Story continues.


SCHF*** Full Year 2023 Financial Highlights:.

PG*** Revenue: Cloud subscription revenue was $304.5 million for the full year 2023, up 29% compared to the full year 2022. Total subscriptions revenue increased 21% year-over-year to $412.3 million for the full year 2023. Professional services revenue was $133.0 million for the full year 2023, compared to $127.8 million for the full year 2022. Total revenue was $545.4 million for the full year 2023, up 17% compared to the full year 2022.

PG*** Operating loss and non-GAAP operating loss: GAAP operating loss was $(108.0) million for the full year 2023, compared to $(145.0) million for the full year 2022. Non-GAAP operating loss was $(54.3) million for the full year 2023, compared to $(83.3) million for the full year 2022.

PG*** Net loss and non-GAAP net loss: GAAP net loss was $(111.4) million for the full year 2023, compared to $(150.9) million for the full year 2022. GAAP net loss per share was $(1.52) for the full year 2023, compared to $(2.08) for the full year 2022. Non-GAAP net loss was $(59.2) million for the full year 2023, compared to $(89.2) million for the full year 2022. Non-GAAP net loss per share was $(0.81) for the full year 2023, compared to the $(1.23) net loss per share for the full year 2022. GAAP and non-GAAP net loss for the full year 2023 included $8.7 million, or $0.12 per share, of foreign currency exchange gains. GAAP and non-GAAP net loss for the full year 2022 included $6.1 million, or $(0.08) per share, of foreign currency exchange losses.

PG*** Adjusted EBITDA : Adjusted EBITDA loss was $(44.8) million for the full year 2023, compared to adjusted EBITDA loss of $(76.0) million for the full year 2022.

CS*** Balance sheet and cash flows: As of December 31, 2023, Appian had total cash, cash equivalents, and investments of $159.0 million. Net cash used in operating activities was $(110.4) million for the full year 2023, compared to $(106.6) million of net cash used in operating activities for the full year 2022.

PG*** A reconciliation of GAAP to non-GAAP financial measures has been provided in the tables following the financial statements in this press release. An explanation of these measures is also included below under the heading “Non-GAAP Financial Measures.”.


SCBQ*** Recent Business Highlights:.

Recent Business Highlights: US Army Revolutionizes Contract Writing with Appian Platform. Appian Government Cloud Achieves “In Process” Designation for FedRAMP High Impact Level.
Appian Delivers Better Business Decisions and Outcomes with AI Plus Data Fabric Analytics. Appian Named a Leader in Everest Group’s Low-code Technology Providers in Insurance PEAK Matrix Assessment 2023.
Appian Named a 2023 Tech100 Honoree by the Northern Virginia Technology Council. 2023 Appian International Partner Award Winners Demonstrate Process Automation Excellence in Europe.
Appian Enhances “One Appian” Global Partner Program Strategy for 2024.


SCG*** Financial Outlook:.

PG*** As of February 15, 2024, guidance for 2024 is as follows:.


First Quarter 2024 Guidance: Cloud subscription revenue is expected to be between $84.0 million and $86.0 million, representing year-over-year growth of 21% to 23%.
Total revenue is expected to be between $148.0 million and $150.0 million, representing a year-over-year increase of 9% to 11%.
Adjusted EBITDA loss is expected to be between $(9.0) million and $(5.0) million.Non-GAAP net loss per share is expected to be between $(0.21) and $(0.16), assuming weighted average common shares outstanding of 73.5 million.


Full Year 2024 Guidance:. Cloud subscription revenue is expected to be between $364.0 million and $366.0 million, representing year-over-year growth of 20%.
Total revenue is expected to be between $615.0 million and $617.0 million, representing a year-over-year increase of 13%.
Adjusted EBITDA loss is expected to be between $(25.0) million and $(20.0) million. Non-GAAP net loss per share is expected to be between $(0.73) and $(0.66), assuming weighted average common shares outstanding of 73.8 million.

"""

In [100]:
# Split into a list of paragraphs
texts = text.split('\n\n')

# Clean up to remove empty spaces and new lines
texts = np.array([t.strip(' \n') for t in texts if t])

In [101]:
# Get the embeddings
response = co.embed(
    texts=texts.tolist(),
).embeddings


default model on embed will be deprecated in the future, please specify a model in the request.


In [102]:
# Check the dimensions of the embeddings
embeds = np.array(response)

# Create the search index, pass the size of embedding
search_index = AnnoyIndex(embeds.shape[1], 'angular')
# Add all the vectors to the search index
for i in range(len(embeds)):
    search_index.add_item(i, embeds[i])

search_index.build(10) # 10 trees
search_index.save('test.ann')

True

In [103]:
def search_andrews_article(query):
    # Get the query's embedding
    query_embed = co.embed(texts=[query]).embeddings
    
    # Retrieve the nearest neighbors
    similar_item_ids = search_index.get_nns_by_vector(query_embed[0],
                                                    10,
                                                  include_distances=True)

    search_results = texts[similar_item_ids[0]]
    
    return search_results

In [104]:
results = search_andrews_article(
    question
)

print(results[0])

default model on embed will be deprecated in the future, please specify a model in the request.


Full Year 2024 Guidance:. Cloud subscription revenue is expected to be between $364.0 million and $366.0 million, representing year-over-year growth of 20%.
Total revenue is expected to be between $615.0 million and $617.0 million, representing a year-over-year increase of 13%.
Adjusted EBITDA loss is expected to be between $(25.0) million and $(20.0) million. Non-GAAP net loss per share is expected to be between $(0.73) and $(0.66), assuming weighted average common shares outstanding of 73.8 million.


In [105]:
def ask_andrews_article(question, num_generations=1):
    
    # Search the text archive
    results = search_andrews_article(question)

    # Get the top result
    context = results[0]

    # Prepare the prompt
    prompt = f"""
    Excerpt from the article titled "How to Build a Career in AI" 
    by Andrew Ng: 
    {context}
    Question: {question}
    
    Extract the answer of the question from the text provided. 
    If the text doesn't contain the answer, 
    reply that the answer is not available."""

    prediction = co.generate(
        prompt=prompt,
        max_tokens=70,
        model="command-nightly",
        temperature=0.5,
        num_generations=num_generations
    )

    return prediction.generations

In [106]:
results = ask_andrews_article(
    question,
    num_generations=3
)

for gen in results:
    print(gen)
    print('--')

default model on embed will be deprecated in the future, please specify a model in the request.


The answer is expected to be between $(25.0) million and $(20.0) million.  It is an expected loss, rather than income. 
--
The expected Adjusted EBITDA income for Appian in Full Year 2024 is an estimated loss of between $(25.0) million and $(20.0) million. 
--
The Full Year 2024 Guidance for Adjusted EBITDA income is expected to be between $(25.0) million and $(20.0) million.  It represents a loss for the company, but with an improving trend as the loss is decreasing year over year. 
--


In [44]:
results = ask_andrews_article(
    "What is the most viewed televised event?",
    num_generations=5
)

default model on embed will be deprecated in the future, please specify a model in the request.


In [45]:
for gen in results:
    print(gen)
    print('--')

The answer is not available. 
--
The answer is not available. 
--
The answer is not available. 
--
The answer is not available. 
--
The answer is not available. 
--
